# Section 3 — Research: Tools for Forecasting in PyTorch

This notebook surveys the key building blocks and libraries used for time-series forecasting with deep learning.

**Topics covered**
1. `nn.LSTM` and `nn.GRU`
2. `nn.Conv1d` with dilation and causal padding (TCN)
3. `nn.TransformerEncoderLayer` and sinusoidal positional encoding
4. NeuralForecast & StatsForecast (Nixtla)
5. Chronos `BaseChronosPipeline.predict_quantiles`
6. Additional concepts: local vs global models, teacher forcing, RevIN, Pinball loss, MASE & WAPE


---
## 1. `nn.LSTM` and `nn.GRU`

### Key constructor parameters

| Parameter | Description |
|-----------|-------------|
| `input_size` | Number of features at each time step |
| `hidden_size` | Dimensionality of the hidden state **H** |
| `num_layers` | Number of stacked recurrent layers |
| `batch_first` | If `True`, tensors are `(B, L, F)`; otherwise `(L, B, F)` |
| `dropout` | Dropout probability between layers (not applied to last layer) |
| `bidirectional` | If `True`, creates a bidirectional variant |

### Output shapes (`batch_first=True`)

```
output : (B, L, H)            – hidden state at every time step
h_n    : (num_layers, B, H)   – final hidden state for each layer
c_n    : (num_layers, B, H)   – final cell state  (LSTM only)
```

For a **GRU** there is no cell state, so the return is `(output, h_n)`.


In [ ]:
import torch
import torch.nn as nn

# ── Hyper-parameters ──────────────────────────────────────────────────────────
B, L, F = 8, 24, 10   # batch, sequence length, input features
H       = 32           # hidden size
LAYERS  = 2

x = torch.randn(B, L, F)

# ── LSTM ──────────────────────────────────────────────────────────────────────
lstm = nn.LSTM(
    input_size=F,
    hidden_size=H,
    num_layers=LAYERS,
    batch_first=True,
    dropout=0.1
)

out_lstm, (h_n, c_n) = lstm(x)
print(f'LSTM  output : {tuple(out_lstm.shape)}')   # (B, L, H)
print(f'LSTM  h_n   : {tuple(h_n.shape)}')         # (LAYERS, B, H)
print(f'LSTM  c_n   : {tuple(c_n.shape)}')         # (LAYERS, B, H)

# ── GRU ───────────────────────────────────────────────────────────────────────
gru = nn.GRU(
    input_size=F,
    hidden_size=H,
    num_layers=LAYERS,
    batch_first=True,
    dropout=0.1
)

out_gru, h_n_gru = gru(x)
print(f'\nGRU   output : {tuple(out_gru.shape)}')
print(f'GRU   h_n   : {tuple(h_n_gru.shape)}')


> **Tip:** To extract only the **last** time-step hidden state use `out_lstm[:, -1, :]` (shape `(B, H)`), which is equivalent to `h_n[-1]` for a single-direction LSTM.


---
## 2. `nn.Conv1d` with Dilation and Causal Padding (TCN)

A **Temporal Convolutional Network (TCN)** stacks dilated causal convolutions to achieve a large receptive field without recurrence.

### Receptive field

For a stack of `n` layers with kernel size `k` and dilation `d_i = 2^i`:

$$\text{RF} = (k - 1) \sum_{i=0}^{n-1} 2^i + 1 = (k-1)(2^n - 1) + 1$$

### Causal padding

To avoid future leakage, pad the **left** side only by `(k-1)*d` and slice away the excess on the right:

```
causal_pad = (kernel_size - 1) * dilation
x_padded   = F.pad(x, (causal_pad, 0))
out        = conv(x_padded)[..., :L]          # trim right
```


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalConv1d(nn.Module):
    """Single dilated causal convolution with same-length output."""

    def __init__(self, in_channels, out_channels, kernel_size, dilation=1):
        super().__init__()
        self.pad = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(
            in_channels, out_channels, kernel_size,
            dilation=dilation, padding=0
        )

    def forward(self, x):               # x: (B, C, L)
        x = F.pad(x, (self.pad, 0))    # pad left
        return self.conv(x)            # output: (B, C_out, L)


class TCNBlock(nn.Module):
    """Residual TCN block: two causal convolutions + skip connection."""

    def __init__(self, channels, kernel_size, dilation):
        super().__init__()
        self.conv1 = CausalConv1d(channels, channels, kernel_size, dilation)
        self.conv2 = CausalConv1d(channels, channels, kernel_size, dilation)
        self.norm1 = nn.LayerNorm(channels)
        self.norm2 = nn.LayerNorm(channels)
        self.relu  = nn.ReLU()

    def forward(self, x):              # x: (B, C, L)
        residual = x
        # Conv expects (B,C,L); LayerNorm on last dim expects (B,L,C)
        h = self.conv1(x).transpose(1, 2)   # (B, L, C)
        h = self.relu(self.norm1(h)).transpose(1, 2)  # (B, C, L)
        h = self.conv2(h).transpose(1, 2)
        h = self.relu(self.norm2(h)).transpose(1, 2)
        return h + residual


# ── Demo ──────────────────────────────────────────────────────────────────────
B, C, L      = 4, 16, 48
kernel_size  = 3
num_blocks   = 4
dilations    = [2**i for i in range(num_blocks)]  # [1,2,4,8]

rf = (kernel_size - 1) * sum(dilations) + 1
print(f'Receptive field: {rf} time steps')

x = torch.randn(B, C, L)
for d in dilations:
    block = TCNBlock(C, kernel_size, dilation=d)
    x = block(x)

print(f'TCN output shape: {tuple(x.shape)}')   # (B, C, L) preserved


---
## 3. `nn.TransformerEncoderLayer` and Sinusoidal Positional Encoding

### Key parameters

| Parameter | Description |
|-----------|-------------|
| `d_model` | Model (embedding) dimension |
| `nhead` | Number of parallel attention heads (`d_model` must be divisible by `nhead`) |
| `dim_feedforward` | Inner dimension of the position-wise FFN |
| `dropout` | Dropout applied inside attention and FFN |
| `batch_first` | If `True`, input is `(B, L, d_model)` |

### Sinusoidal positional encoding

$$PE_{(pos,\,2i)} = \sin\!\left(\frac{pos}{10000^{2i/d}}\right), \quad
PE_{(pos,\,2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)$$


In [ ]:
import math
import torch
import torch.nn as nn


class SinusoidalPE(nn.Module):
    """Adds fixed sinusoidal positional encoding to the input."""

    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)               # (max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1)      # (max_len, 1)
        div = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )                                                 # (d_model/2,)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        pe = pe.unsqueeze(0)                              # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):                                # x: (B, L, d_model)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


# ── Demo ──────────────────────────────────────────────────────────────────────
B, L, d_model = 4, 48, 64

pos_enc  = SinusoidalPE(d_model, dropout=0.0)
enc_layer = nn.TransformerEncoderLayer(
    d_model=d_model,
    nhead=8,
    dim_feedforward=256,
    dropout=0.1,
    batch_first=True
)
encoder = nn.TransformerEncoder(enc_layer, num_layers=3)

x   = torch.randn(B, L, d_model)
x   = pos_enc(x)
out = encoder(x)

print(f'Encoder input  shape: {tuple(x.shape)}')    # (B, L, d_model)
print(f'Encoder output shape: {tuple(out.shape)}')  # (B, L, d_model)


---
## 4. NeuralForecast & StatsForecast (Nixtla)

Both libraries expect a **long-format** DataFrame with exactly three columns:

| Column | Type | Description |
|--------|------|-------------|
| `unique_id` | str / int | Series identifier |
| `ds` | datetime | Timestamp |
| `y` | float | Target value |

### Cross-validation parameters

| Parameter | Description |
|-----------|-------------|
| `n_windows` | Number of back-test windows |
| `step_size` | Steps between consecutive windows (default = `h`) |
| `refit` | Retrain model for each window if `True` |

> **Note:** The examples below show the API calls. Running them requires the optional `neuralforecast` and `statsforecast` packages installed in your environment.


In [ ]:
# ── Minimal NeuralForecast example ────────────────────────────────────────────
# pip install neuralforecast

try:
    import pandas as pd
    import numpy as np
    from neuralforecast import NeuralForecast
    from neuralforecast.models import NHITS

    # Synthetic data in Nixtla long format
    T   = 200
    df  = pd.DataFrame({
        'unique_id': 'series_1',
        'ds'       : pd.date_range('2020-01-01', periods=T, freq='D'),
        'y'        : np.sin(np.linspace(0, 8 * np.pi, T)) + np.random.randn(T) * 0.1,
    })

    H   = 14   # forecast horizon
    nf  = NeuralForecast(
        models=[NHITS(h=H, input_size=2*H, max_steps=100)],
        freq='D'
    )

    # Cross-validation
    cv_df = nf.cross_validation(
        df        = df,
        n_windows = 3,
        step_size = H,
        refit     = False
    )
    print(cv_df.head())

except ImportError:
    print('neuralforecast not installed — showing API structure only.')

    EXAMPLE = '''
    from neuralforecast import NeuralForecast
    from neuralforecast.models import NHITS

    nf = NeuralForecast(models=[NHITS(h=14, input_size=28, max_steps=500)], freq="D")
    nf.fit(df)                                   # df: unique_id, ds, y
    preds = nf.predict()                          # returns DataFrame

    cv = nf.cross_validation(df, n_windows=3, step_size=14, refit=False)
    '''
    print(EXAMPLE)


In [ ]:
# ── Minimal StatsForecast example ─────────────────────────────────────────────
# pip install statsforecast

try:
    import pandas as pd
    import numpy as np
    from statsforecast import StatsForecast
    from statsforecast.models import AutoETS, AutoARIMA

    T   = 200
    df  = pd.DataFrame({
        'unique_id': 'series_1',
        'ds'       : pd.date_range('2020-01-01', periods=T, freq='D'),
        'y'        : np.sin(np.linspace(0, 8 * np.pi, T)) + np.random.randn(T) * 0.1,
    })

    sf  = StatsForecast(models=[AutoETS(season_length=7), AutoARIMA()], freq='D')
    cv_df = sf.cross_validation(
        df        = df,
        h         = 14,
        n_windows = 3,
        step_size = 14
    )
    print(cv_df.head())

except ImportError:
    print('statsforecast not installed — showing API structure only.')

    EXAMPLE = '''
    from statsforecast import StatsForecast
    from statsforecast.models import AutoETS

    sf = StatsForecast(models=[AutoETS(season_length=7)], freq="D")
    sf.fit(df)
    preds = sf.predict(h=14)

    cv = sf.cross_validation(df, h=14, n_windows=3, step_size=14)
    '''
    print(EXAMPLE)


---
## 5. Chronos — `BaseChronosPipeline.predict_quantiles`

[Chronos](https://github.com/amazon-science/chronos-forecasting) is a pre-trained probabilistic forecasting model by Amazon.

### Input format
A **list of 1-D `torch.Tensor`** objects (one per time series), each of shape `(T,)`.  
Missing values should be represented as `torch.nan`.

### Key `predict_quantiles` arguments

| Argument | Description |
|----------|-------------|
| `context` | List of `(T,)` tensors — historical observations |
| `prediction_length` | Number of future steps to forecast |
| `quantile_levels` | List of quantiles, e.g. `[0.1, 0.5, 0.9]` |

Returns a tensor of shape `(N, Q, H)` where N = batch size, Q = number of quantiles, H = prediction length.


In [ ]:
# pip install chronos-forecasting

try:
    import torch
    from chronos import BaseChronosPipeline

    pipeline = BaseChronosPipeline.from_pretrained(
        'amazon/chronos-t5-tiny',   # smallest variant for demo
        device_map='cpu',
        torch_dtype=torch.float32
    )

    # Synthetic context: two time series of different lengths
    context = [
        torch.randn(100),
        torch.randn(150)
    ]

    quantile_levels = [0.1, 0.5, 0.9]
    prediction_length = 12

    quantiles, mean = pipeline.predict_quantiles(
        context          = context,
        prediction_length= prediction_length,
        quantile_levels  = quantile_levels
    )

    print(f'Quantiles shape : {tuple(quantiles.shape)}')  # (N, Q, H)
    print(f'Mean shape      : {tuple(mean.shape)}')       # (N, H)

except ImportError:
    print('chronos-forecasting not installed — showing API structure only.')

    EXAMPLE = '''
    from chronos import BaseChronosPipeline
    import torch

    pipeline = BaseChronosPipeline.from_pretrained("amazon/chronos-t5-small")

    context = [torch.tensor(series_values)]  # list of 1-D tensors

    quantiles, mean = pipeline.predict_quantiles(
        context           = context,
        prediction_length = 24,
        quantile_levels   = [0.1, 0.5, 0.9]
    )
    # quantiles: (N, Q, H)  |  mean: (N, H)
    '''
    print(EXAMPLE)


---
## 6. Additional Concepts

### 6.1 Local vs Global Models

| Dimension | Local Model | Global Model |
|-----------|-------------|---------------|
| Training data | One model **per series** | Single model trained on **all series** |
| Parameters | `O(N × P)` where N = # series | `O(P)` shared |
| Cold start | Cannot forecast unseen series | Can generalise to new series |
| Examples | ARIMA, ETS | N-BEATS, TFT, Chronos |

**Global models** leverage cross-series patterns and scale better with data, but require careful normalisation to handle series with different scales.


---
### 6.2 Teacher Forcing in Seq2Seq

In an encoder–decoder architecture the decoder auto-regressively generates predictions one step at a time.

**Without teacher forcing** — the decoder receives its own previous prediction as input:

```
Encoder:  [x1, x2, ..., xT]  →  context vector  c
Decoder:  [<START>]  →  ŷ1  →  ŷ2  →  ...  →  ŷH
                 ↑ ŷ_{t-1} fed back
```

**With teacher forcing** — the ground-truth is fed as decoder input at each step:

```
Encoder:  [x1, x2, ..., xT]  →  c
Decoder:  [y0, y1, ..., y_{H-1}]  →  [ŷ1, ŷ2, ..., ŷH]
                  ↑ ground-truth fed in (teacher)
```

**Trade-off:** Teacher forcing speeds up training and stabilises gradients but creates an **exposure bias** — at inference time the model never saw its own errors as inputs. A common fix is **scheduled sampling**: start with full teacher forcing and gradually switch to model predictions during training.


---
### 6.3 RevIN — Reversible Instance Normalisation

**RevIN** (Kim et al., 2022) addresses *distribution shift* and *level shifts* between the look-back window and the forecast horizon.

**Algorithm**
1. **Normalise** the input window: subtract its mean, divide by its std.
2. Apply the forecasting model on normalised inputs.
3. **Denormalise** the output using the *same* statistics (affine transform with learnable `γ` and `β`).

This ensures the model does not need to memorise absolute levels, only patterns — making it far more robust to sudden level changes in non-stationary series.


In [ ]:
import torch
import torch.nn as nn


class RevIN(nn.Module):
    """
    Reversible Instance Normalisation (Kim et al., 2022).

    Args:
        num_features : number of channels / variates
        eps          : numerical stability constant
        affine       : learn per-channel γ and β
    """

    def __init__(self, num_features: int, eps: float = 1e-5, affine: bool = True):
        super().__init__()
        self.num_features = num_features
        self.eps          = eps
        self.affine       = affine

        if affine:
            self.gamma = nn.Parameter(torch.ones(1, 1, num_features))
            self.beta  = nn.Parameter(torch.zeros(1, 1, num_features))

    def _get_stats(self, x):            # x: (B, L, C)
        self.mean = x.mean(dim=1, keepdim=True)                      # (B,1,C)
        self.std  = x.std(dim=1, keepdim=True, unbiased=False) + self.eps

    def normalize(self, x):             # forward pass: standardise
        self._get_stats(x)
        x = (x - self.mean) / self.std
        if self.affine:
            x = x * self.gamma + self.beta
        return x

    def denormalize(self, x):           # inverse: undo normalisation
        if self.affine:
            x = (x - self.beta) / self.gamma
        return x * self.std + self.mean

    def forward(self, x, mode='norm'):  # mode: 'norm' | 'denorm'
        if mode == 'norm':
            return self.normalize(x)
        elif mode == 'denorm':
            return self.denormalize(x)
        raise ValueError(f'Unknown mode: {mode}')


# ── Demo ──────────────────────────────────────────────────────────────────────
B, L, C = 4, 48, 7
revin   = RevIN(num_features=C)

x_raw  = torch.randn(B, L, C) * 100 + 500   # large offset to show level shift
x_norm = revin(x_raw, mode='norm')
print(f'Normalised mean  ≈ {x_norm.mean().item():.4f}  (should be ≈ 0)')
print(f'Normalised std   ≈ {x_norm.std().item():.4f}   (should be ≈ 1)')

# Simulate model output (same shape as normalised)
y_norm = x_norm + 0.01 * torch.randn_like(x_norm)

# Denormalise to get back to original scale
y_hat  = revin(y_norm, mode='denorm')
print(f'Denormalised mean ≈ {y_hat.mean().item():.1f}  (should be ≈ 500)')


---
### 6.4 Pinball Loss (Quantile Loss)

The **pinball loss** (quantile loss) trains a model to predict a specific quantile `q ∈ (0,1)`:

$$L_q(y, \hat{y}) = q \cdot \max(y - \hat{y},\, 0) + (1 - q) \cdot \max(\hat{y} - y,\, 0)$$

- For `q = 0.9`: over-prediction is cheap, under-prediction is expensive → model learns the 90th percentile.
- The median (`q = 0.5`) corresponds to the **MAE** loss.


In [ ]:
import torch
import torch.nn as nn


class PinballLoss(nn.Module):
    """
    Pinball (quantile) loss.

    Args:
        quantile : target quantile level q ∈ (0, 1)
    """

    def __init__(self, quantile: float = 0.5):
        super().__init__()
        assert 0 < quantile < 1, 'quantile must be in (0, 1)'
        self.q = quantile

    def forward(self, y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
        errors = y_true - y_pred
        loss   = torch.max(self.q * errors, (self.q - 1) * errors)
        return loss.mean()


# ── Functional version ────────────────────────────────────────────────────────
def pinball_loss(y_pred: torch.Tensor, y_true: torch.Tensor, q: float) -> torch.Tensor:
    errors = y_true - y_pred
    return torch.max(q * errors, (q - 1) * errors).mean()


# ── Demo ──────────────────────────────────────────────────────────────────────
torch.manual_seed(0)
y_true = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
y_pred = torch.tensor([1.5, 1.8, 3.2, 3.9, 5.5])

for q in [0.1, 0.5, 0.9]:
    criterion = PinballLoss(quantile=q)
    loss_val  = criterion(y_pred, y_true)
    print(f'Pinball loss (q={q:.1f}): {loss_val.item():.4f}')


---
### 6.5 Scale-Free Metrics: MASE and WAPE

Standard metrics like MAE and RMSE are scale-dependent and cannot be averaged across series with different magnitudes. **Scale-free** metrics solve this.

#### Mean Absolute Scaled Error (MASE)

$$\text{MASE} = \frac{\text{MAE}}{\text{MAE}_{\text{naive}}} = \frac{\frac{1}{H}\sum_{t=1}^{H}|y_t - \hat{y}_t|}{\frac{1}{T-s}\sum_{t=s+1}^{T}|y_t - y_{t-s}|}$$

where `s` is the seasonal period (s=1 for non-seasonal data). MASE < 1 means the model beats the seasonal naïve baseline.

#### Weighted Absolute Percentage Error (WAPE)

$$\text{WAPE} = \frac{\sum_{t=1}^{H}|y_t - \hat{y}_t|}{\sum_{t=1}^{H}|y_t|}$$

WAPE is equivalent to a **weighted MAE** normalised by the sum of actuals. Unlike MAPE, it is stable when `y` is near zero (no division by individual values).


In [ ]:
import torch


def mase(
    y_pred    : torch.Tensor,   # (H,) or (B, H)
    y_true    : torch.Tensor,   # (H,) or (B, H)  – forecast horizon
    y_train   : torch.Tensor,   # (T,) or (B, T)  – training history
    seasonality: int = 1
) -> torch.Tensor:
    """
    Mean Absolute Scaled Error.
    MASE = MAE(forecast) / MAE(seasonal-naive on training set)
    """
    mae_forecast = (y_true - y_pred).abs().mean(dim=-1)
    # Seasonal naïve error on the training period
    naive_errors = (y_train[..., seasonality:] - y_train[..., :-seasonality]).abs()
    mae_naive    = naive_errors.mean(dim=-1)
    return (mae_forecast / (mae_naive + 1e-8)).mean()


def wape(
    y_pred: torch.Tensor,       # (H,) or (B, H)
    y_true: torch.Tensor        # (H,) or (B, H)
) -> torch.Tensor:
    """
    Weighted Absolute Percentage Error.
    WAPE = sum|y - ŷ| / sum|y|
    """
    return (y_true - y_pred).abs().sum(dim=-1) / (y_true.abs().sum(dim=-1) + 1e-8)


# ── Demo ──────────────────────────────────────────────────────────────────────
torch.manual_seed(42)

T, H     = 100, 12
y_train  = torch.cumsum(torch.randn(T), dim=0)           # random walk
y_true_f = y_train[-1] + torch.randn(H)                  # true future
y_pred_f = y_train[-1] + torch.randn(H) * 0.5            # model forecast

print(f'MASE : {mase(y_pred_f, y_true_f, y_train).item():.4f}')
print(f'WAPE : {wape(y_pred_f, y_true_f).item():.4f}')


---
## Summary

| Tool / Concept | Key Takeaway |
|----------------|--------------|
| `nn.LSTM` / `nn.GRU` | Gated RNNs; output `(B,L,H)`, states `(layers,B,H)` |
| TCN + causal padding | Left-pad by `(k-1)·d`; RF grows exponentially with dilation |
| Transformer | Self-attention + sinusoidal PE; no recurrence |
| NeuralForecast | `unique_id / ds / y` format; `cross_validation(n_windows, step_size, refit)` |
| Chronos | Zero-shot probabilistic; `predict_quantiles(context, H, q_levels)` |
| Local vs Global | Local = one model / series; Global = shared across series |
| Teacher forcing | Ground-truth decoder input during training; scheduled sampling fixes exposure bias |
| RevIN | Normalise-predict-denormalise; robust to level shifts |
| Pinball loss | `q·max(y-ŷ,0) + (1-q)·max(ŷ-y,0)`; trains quantile models |
| MASE / WAPE | Scale-free; compare across series and baselines |
